# Merging new SOC results
This notebook is to merge all new results generated using the new FAO GAEZ yields as inputs, including reduced tillage and just crops.

## SETUP

### Modules

In [1]:
import pandas as pd
import polars as pl
import geopandas as gpd
import sbtn_leaf.map_plotting as mp
import sbtn_leaf.map_calculations as mc
import sbtn_leaf.paths as sbtn_path
import sqlite3

LEAFS_FOLDER = sbtn_path._LEAFS_DIR
PROJECT_ROOT = sbtn_path.project_root()

Could not determine dtype for column 1, falling back to string


### Data

Data paths

In [2]:
# Geopakcages
gpckg_og_ctr_path   = LEAFS_FOLDER / "SOC/SOC_2030_country.gpkg"
gpckg_og_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_subcountry.gpkg"
gpckg_og_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_ecoregion.gpkg"

gpckg_gaez_ctr_path   = LEAFS_FOLDER / "SOC_2030_country_crops_clipped.gpkg"
gpckg_gaez_sc_path    = LEAFS_FOLDER / "SOC_2030_subcountry_crops_clipped.gpkg"
gpckg_gaez_er_path    = LEAFS_FOLDER / "SOC_2030_ecoregions_crops_clipped.gpkg"

gpckg_rt_ctr_path   = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_country.gpkg"
gpckg_rt_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_subcountry.gpkg"
gpckg_rt_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_ecoregions.gpkg"

#gpcks layers names
ctr_layer = "soc_leaf_country"
sc_layer = "soc_leaf_subcountry"
er_layer = "soc_leaf_ecoregions"
geom_layer = "geometry_layer"

# csvs
csv_og_ctr_path   = LEAFS_FOLDER / "SOC/SOC_2030_country.csv"
csv_og_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_subcountry.csv"
csv_og_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_ecoregion.csv"

csv_gaez_ctr_path   = LEAFS_FOLDER / "SOC_2030_country_crops_clipped.csv"
csv_gaez_sc_path    = LEAFS_FOLDER / "SOC_2030_subcountry_crops_clipped.csv"
csv_gaez_er_path    = LEAFS_FOLDER / "SOC_2030_ecoregions_crops_clipped.csv"

csv_rt_ctr_path   = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_country.csv"
csv_rt_sc_path    = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_subcountry.csv"
csv_rt_er_path    = LEAFS_FOLDER / "SOC/SOC_2030_reducedtillage_ecoregions.csv"

### Opening geopackages

Countries

In [50]:
og_country_gpckg = gpd.read_file(gpckg_og_ctr_path, layer=geom_layer)
og_country_df = pd.read_csv(csv_og_ctr_path)
og_country_gdf = og_country_gpckg.merge(og_country_df.drop(columns="country"), how="left", on="ADM0_NAME")

In [51]:
gaez_country_gpckg = gpd.read_file(gpckg_gaez_ctr_path, layer=ctr_layer)
gaez_country_geom =  gpd.read_file(gpckg_gaez_ctr_path, layer=geom_layer)
gaez_country_gdf = gaez_country_geom.merge(gaez_country_gpckg.drop(columns="_source_file"), how="left", on="ADM0_NAME")

In [52]:
rt_country_gpckg = gpd.read_file(gpckg_rt_ctr_path, layer=ctr_layer)
rt_country_geom =  gpd.read_file(gpckg_rt_ctr_path, layer=geom_layer)
rt_country_gdf = rt_country_geom.merge(rt_country_gpckg.drop(columns="_source_file"), how="left", on="ADM0_NAME")

Subcountries

In [3]:
og_subcountry_geom =   gpd.read_file(gpckg_og_sc_path, layer=geom_layer)
og_subcountry_values = gpd.read_file(gpckg_og_sc_path, layer=sc_layer)
og_subcountry_values = og_subcountry_values.drop(columns = "_source_file")

In [4]:
gaez_subcountry_geom =  gpd.read_file(gpckg_gaez_sc_path, layer=geom_layer)
gaez_subcountry_values = gpd.read_file(gpckg_gaez_sc_path, layer=sc_layer)
gaez_subcountry_values = gaez_subcountry_values.drop(columns = "_source_file")

In [5]:
rt_subcountry_geom =  gpd.read_file(gpckg_rt_sc_path, layer=geom_layer)
rt_subcountry_values = gpd.read_file(gpckg_rt_sc_path, layer=sc_layer)
rt_subcountry_values = rt_subcountry_values.drop(columns = "_source_file")

Ecoregions

In [29]:
og_er_geom =   gpd.read_file(gpckg_og_er_path, layer=geom_layer)
og_er_values = gpd.read_file(gpckg_og_er_path, layer=er_layer)
og_er_values = og_er_values.drop(columns = "_source_file")

In [30]:
gaez_er_geom =  gpd.read_file(gpckg_gaez_er_path, layer=geom_layer)
gaez_er_values = gpd.read_file(gpckg_gaez_er_path, layer=er_layer)
gaez_er_values = gaez_er_values.drop(columns = "_source_file")

In [31]:
rt_er_geom =  gpd.read_file(gpckg_rt_er_path, layer=geom_layer)
rt_er_values = gpd.read_file(gpckg_rt_er_path, layer=er_layer)
rt_er_values = rt_er_values.drop(columns = "_source_file")

## Merging
So there are 3 things to be done in the merger: 1) Detect all crops flows, 2) Exchange those crop flows for the new GAEZ ones, and 3) Add the reduced tillage ones.

### Countries

All the crops present in the GAEZ files needs to be exchanged, so a list will be created, then eliminated from the original geopackage, and then replaced by the GAEZ ones.

In [53]:
gaez_country_flows_list = gaez_country_gpckg["flow_name"].unique()

In [54]:
non_crops_og_country_df = og_country_df[~og_country_df["flow_name"].isin(gaez_country_flows_list)].drop(columns = "country")

So there are a couples of things that still need to be filtered out, including West Bank results, Maize, rainfed. 

In [55]:
non_crops_og_country_df = non_crops_og_country_df[(non_crops_og_country_df["ADM0_NAME"] != "West Bank") & (non_crops_og_country_df["flow_name"]!= "Maize_rf_2030y_SOC")] 

Now, because of some weird reason the geopackage of the country leaf layer doesn't exist, so need to work with the csv, which is in long format, which needs to be transformed into wide format.

In [56]:
non_crops_og_country_df_wide = non_crops_og_country_df.pivot(values = "value", columns="metric", index=["ADM0_NAME", "flow_name"]).reset_index().rename(columns={"cf_mean": "cf"})

Dropping the _source_file column of the other 2 geopackages

In [57]:
gaez_country_gpckg = gaez_country_gpckg.drop(columns = "_source_file")
rt_country_gpckg = rt_country_gpckg.drop(columns = "_source_file")

Stacking everything

In [58]:
final_country_df = pd.concat([non_crops_og_country_df_wide, gaez_country_gpckg, rt_country_gpckg], ignore_index= True)

Finally saving the file

In [16]:
country_final_gpckg_path = LEAFS_FOLDER / "SOC/SOC_2030_country_v1.0.gpkg"

In [17]:
og_country_gpckg.to_file(country_final_gpckg_path, layer = geom_layer, driver="GPKG")

In [ ]:
# Write values table directly into the geopackage (which is just a sqlite db)
conn = sqlite3.connect(country_final_gpckg_path)
final_country_df.to_sql(ctr_layer, conn, if_exists="replace", index=False)
conn.close()

### Subcountry

In [6]:
gaez_subcountry_flows_list = gaez_subcountry_values["flow_name"].unique()

In [7]:
non_crops_og_subcountry_values = og_subcountry_values[~og_subcountry_values["flow_name"].isin(gaez_subcountry_flows_list)]

In [8]:
og_subcountry_flows = og_subcountry_values["flow_name"].unique()

So interesting discovery.... the subcountry averages do not have any forestry or grassland flows, so running those now

In [9]:
# Inputs that stay the same
input_folder = LEAFS_FOLDER / "SOC/rasters"
output_folder = LEAFS_FOLDER/"SOC/"
cf_name = "SOC_2030"
cf_unit = "t C/ha"
reset_gpck = True

In [10]:
# Inputs that change by area type
area_type = "subcountry"
layer_name = "soc_leaf_subcountry"
sc_shp = gpd.read_file(PROJECT_ROOT/'data/CountryLayers/SubCountry_Level1/g2015_2014_1.shp')
master_key = "ADM1_CODE"
result_key = "ADM1_CODE"

Running the averages

In [11]:
gpkg_path_sc, df_results_sc = mc.build_cfs_gpkg_from_rasters(
    gpckg_name="SOC_2030_subcountry_forest_grass",
    input_folder = str(input_folder),
    output_folder = str(output_folder),
    layer_name=layer_name,
    master_gdf=sc_shp,                  # e.g., countries, subcountries, or ecoregions
    master_key=master_key,              # or 'ISO_A3', 'ADM1_CODE', 'ECO_NAME'
    result_key=result_key,              # must match the column emitted by your calc gdf
    cf_name=cf_name,
    cf_unit=cf_unit,
    area_type=area_type,
    calc_kwargs=dict(raster_band = 15, outlier_method="log1p_win", q_low = 0.01, q_high = 0.99),     # Chooses the raster band. Due to outliers in crops, setting it to windsorize, replacing all values outside 1, 99 percentiles with those values.
    reset_gpkg=True,
    run_test=False, 
    max_workers=3
)

print(f"Wrote {df_results_sc.shape[0]} rows into {gpkg_path_sc}")

2026-04-02 12:06:13,175 - INFO - Building 'soc_leaf_subcountry' from rasters in C:\Users\loyola\OneDrive - World Wildlife Fund, Inc\Documents\203. Python projects\SBTN_Test\LEAFs\SOC\rasters into C:\Users\loyola\OneDrive - World Wildlife Fund, Inc\Documents\203. Python projects\SBTN_Test\LEAFs\SOC (C:\Users\loyola\OneDrive - World Wildlife Fund, Inc\Documents\203. Python projects\SBTN_Test\LEAFs\SOCSOC_2030_subcountry_forest_grass.gpkg)



Processing rasters (soc_leaf_subcountry):   0%|          | 0/21 [00:00<?, ?raster/s]

2026-04-02 12:06:44,087 - INFO - Calculating SOC_2030 for BRDC_Boreal dry_2030y_SOC...
Calculating SOC_2030 for BRDC_Boreal dry_2030y_SOC...2026-04-02 12:06:44,092 - INFO - Calculating SOC_2030 for BRDC_Boreal moist_2030y_SOC...

2026-04-02 12:06:44,094 - INFO - Calculating SOC_2030 for BRDC_Cold temperate dry_2030y_SOC...
Calculating SOC_2030 for BRDC_Boreal moist_2030y_SOC...


Processing rasters (soc_leaf_subcountry):   0%|          | 0/21 [00:00<?, ?raster/s]

Calculating SOC_2030 for BRDC_Cold temperate dry_2030y_SOC...
2026-04-02 12:09:14,526 - INFO - Calculating SOC_2030 for BRDC_Cold temperate moist_2030y_SOC...
Calculating SOC_2030 for BRDC_Cold temperate moist_2030y_SOC...
2026-04-02 12:09:20,653 - INFO - Calculating SOC_2030 for BRDC_Subtropical_2030y_SOC...
Calculating SOC_2030 for BRDC_Subtropical_2030y_SOC...
2026-04-02 12:09:41,512 - INFO - Calculating SOC_2030 for BRDC_Tropical_2030y_SOC...
Calculating SOC_2030 for BRDC_Tropical_2030y_SOC...
2026-04-02 12:10:30,085 - INFO - Calculating SOC_2030 for BRDC_Warm temperate dry_2030y_SOC...
Calculating SOC_2030 for BRDC_Warm temperate dry_2030y_SOC...
2026-04-02 12:11:23,879 - INFO - Calculating SOC_2030 for BRDC_Warm temperate moist_2030y_SOC...
Calculating SOC_2030 for BRDC_Warm temperate moist_2030y_SOC...
2026-04-02 12:12:13,906 - INFO - Calculating SOC_2030 for NEEV_Boreal dry_2030y_SOC...
Calculating SOC_2030 for NEEV_Boreal dry_2030y_SOC...
2026-04-02 12:12:37,555 - INFO - Calcu

Wrote 215586 rows into C:\Users\loyola\OneDrive - World Wildlife Fund, Inc\Documents\203. Python projects\SBTN_Test\LEAFs\SOCSOC_2030_subcountry_forest_grass.gpkg


So now that's done, the data can be loaded

In [13]:
forest_grass_subcountry_geom =   gpd.read_file(gpkg_path_sc, layer=geom_layer)
forest_grass_subcountry_values = gpd.read_file(gpkg_path_sc, layer=sc_layer)
forest_grass_subcountry_values = forest_grass_subcountry_values.drop(columns = "_source_file")

And the gpckg created

In [22]:
final_subcountry_df = pd.concat([forest_grass_subcountry_values, gaez_subcountry_values, rt_subcountry_values], ignore_index= True)

Storing the file

In [24]:
subcountry_final_gpckg_path = LEAFS_FOLDER / "SOC/SOC_2030_subcountry_v1.0.gpkg"

In [25]:
og_subcountry_geom.to_file(subcountry_final_gpckg_path, layer = geom_layer, driver="GPKG")

In [26]:
# Write values table directly into the geopackage (which is just a sqlite db)
conn = sqlite3.connect(subcountry_final_gpckg_path)
final_subcountry_df.to_sql(sc_layer, conn, if_exists="replace", index=False)
conn.close()

### Ecoregions

In [32]:
gaez_er_flows_list = gaez_er_values["flow_name"].unique()

In [34]:
non_crops_og_er_values = og_er_values[~og_er_values["flow_name"].isin(gaez_er_flows_list)]

Stacking everything

In [85]:
final_er_df = pd.concat([non_crops_og_er_values, gaez_er_values, rt_er_values], ignore_index= True)

Removing Antartica

In [92]:
antartica_adm1codes = og_er_geom[og_er_geom["REALM"]=="Antarctica"]["ECO_ID"].unique()

In [94]:
final_er_df = final_er_df[~final_er_df["ECO_ID"].isin(antartica_adm1codes)]

Finally saving the file

In [95]:
er_final_gpckg_path = LEAFS_FOLDER / "SOC/SOC_2030_ecoregion_v1.0.gpkg"

In [96]:
og_er_geom.to_file(er_final_gpckg_path, layer = geom_layer, driver="GPKG")

In [97]:
# Write values table directly into the geopackage (which is just a sqlite db)
conn = sqlite3.connect(er_final_gpckg_path)
final_er_df.to_sql(er_layer, conn, if_exists="replace", index=False)
conn.close()

## Storing csv's
So all geopackages have been created, csv can be stored.

In [48]:
country_final_csv_path = LEAFS_FOLDER / "SOC/SOC_2030_country_v1.0.csv"
subcountry_final_csv_path = LEAFS_FOLDER / "SOC/SOC_2030_subcountry_v1.0.csv"
ecoregion_final_csv_path = LEAFS_FOLDER / "SOC/SOC_2030_ecoregion_v1.0.csv"

Country

In [ ]:
final_country_df.to_csv(country_final_csv_path, sep=";", index=False)

Subcountry and Ecoregion needs a little bit more information to be useful by itself, so need to attach those descriptors first

In [64]:
final_subcountry_df=final_subcountry_df.merge(og_subcountry_geom[["ADM1_CODE","ADM1_NAME","ADM0_NAME"]], how="left", on= "ADM1_CODE")

In [66]:
leaf_columns = ["flow_name", "cf", "cf_median", "cf_std"]

In [67]:
final_subcountry_df = final_subcountry_df[["ADM0_NAME", "ADM1_NAME","ADM1_CODE"]+leaf_columns]

In [72]:
final_subcountry_df = final_subcountry_df.rename(columns={"cf":"leaf", "cf_median": "leaf_median", "cf_std": "leaf_std"})

In [74]:
final_subcountry_df.to_csv(subcountry_final_csv_path, sep=";", index=False)

Now Ecoregions

In [76]:
final_er_df=final_er_df.merge(og_er_geom[["ECO_NAME", "BIOME_NUM", "BIOME_NAME", "REALM", "ECO_BIOME_", "ECO_ID"]], how="left", on= "ECO_ID")

In [79]:
final_er_df = final_er_df[["REALM", "BIOME_NAME",  "BIOME_NUM",  "ECO_BIOME_","ECO_NAME", "ECO_ID"]+leaf_columns]

Filtering out antartica...

In [81]:
final_er_df = final_er_df[final_er_df["REALM"] != "Antarctica"]

Renaming

In [83]:
final_er_df = final_er_df.rename(columns={"cf":"leaf", "cf_median": "leaf_median", "cf_std": "leaf_std"})

In [84]:
final_er_df.to_csv(ecoregion_final_csv_path, sep=";", index=False)